In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('KaggleV2-May-2016.csv')

# Data Cleaning

In [3]:
df.rename(columns={'Hipertension': 'Hypertension', 'Handcap': 'Handicap', 'No-show': 'NoShow'}, inplace=True)

In [4]:
df = df[df['Age'] >= 0]

# Feature Engineering

In [5]:
# Convert date strings to datetime objects
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

In [6]:
# Calculate WaitDays (Time between booking and appointment)
df['WaitDays'] = (pd.to_datetime(df['AppointmentDay'].dt.date) - pd.to_datetime(df['ScheduledDay'].dt.date)).dt.days

In [7]:
# Drop data errors where appointment was before scheduling
df = df[df['WaitDays'] >= 0]

In [8]:
# Extract Appointment Day of Week
df['DayOfWeek'] = df['AppointmentDay'].dt.day_name()

# Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder
le_gender = LabelEncoder()
df['Gender_Enc'] = le_gender.fit_transform(df['Gender'])

le_dow = LabelEncoder()
df['DayOfWeek_Enc'] = le_dow.fit_transform(df['DayOfWeek'])

df['NoShow_Enc'] = df['NoShow'].apply(lambda x: 1 if x == 'Yes' else 0)

features = ['Age', 'Gender_Enc', 'Scholarship', 'Hypertension', 'Diabetes',
            'Alcoholism', 'Handicap', 'SMS_received', 'WaitDays', 'DayOfWeek_Enc']
X = df[features]
y = df['NoShow_Enc']

# Model Training

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=6, random_state=42)

# Evaluation

In [11]:
y_pred = dt_model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 0.7967

Classification Report:
               precision    recall  f1-score   support

           0       0.80      1.00      0.89     17621
           1       0.45      0.01      0.02      4484

    accuracy                           0.80     22105
   macro avg       0.63      0.50      0.45     22105
weighted avg       0.73      0.80      0.71     22105



# Feature Importance

In [12]:
feat_importance = pd.DataFrame({'Feature': features, 'Importance': dt_model.feature_importances_})
print("\nTop Predictors:\n", feat_importance.sort_values(by='Importance', ascending=False))


Top Predictors:
          Feature  Importance
8       WaitDays    0.818504
0            Age    0.129715
7   SMS_received    0.024205
2    Scholarship    0.006315
6       Handicap    0.006009
5     Alcoholism    0.005996
1     Gender_Enc    0.005919
9  DayOfWeek_Enc    0.001407
4       Diabetes    0.001029
3   Hypertension    0.000902


In [13]:
df.to_csv('Cleaned_Appointments_for_PowerBI.csv', index=False)

In [14]:
import pickle

model_filename = 'healthcare_noshow_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(dt_model, file)